# Heterogeneous Cluster - a hello world training job


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

---


This basic example shows how to run a Heterogeneous Clusters training job consisting of two instance groups. Each instance group includes a different instance type. Each instance prints its environment information including its instance group and exits.

This notebook uses the SageMaker Python SDK **v3** `ModelTrainer` interface. Heterogeneous clusters are configured through the `Compute` configuration object by passing a list of `InstanceGroup` objects.

You can retrieve environment information in either of the following ways:
  - **Option 1**: Read instance group information using the convenient `sagemaker_training.environment.Environment` class.
  - **Option 2**: Read instance group information from `/opt/ml/input/config/resourceconfig.json`.
 
 
Note: This notebook does not demonstrate offloading of data preprocessing job to data group and deep neural network training to dnn_group. We will cover those examples in [TensorFlow's tf.data.service based Amazon SageMaker Heterogeneous Clusters for training](../tf.data.service.sagemaker/hetero-tensorflow-restnet50.ipynb) and [PyTorch and gRPC distributed dataloader based Amazon SageMaker Heterogeneous Clusters for training](../pt.grpc.sagemaker/hetero-pytorch-mnist.ipynb) notebooks.

### A. Setting up SageMaker Studio notebook
#### Before you start
Ensure you have selected Python 3 image for your SageMaker Studio Notebook instance, and running on _ml.t3.medium_ instance type.

#### Step 1 - Install the SageMaker Python SDK v3 and dependent packages
Heterogeneous Clusters for Amazon SageMaker model training requires the SageMaker Python SDK v3 and up-to-date boto3 client libraries.

In [1]:
# [papermill-run] pip install disabled; packages preinstalled in exec env
print('pip install skipped for papermill run')


pip install skipped for papermill run


#### Step 2 - Restart the notebook kernel 

In [2]:
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

#### Step 3 - Validate SageMaker Python SDK version
Ensure the output of the cell below reflects:

- SageMaker Python SDK version 3.0.0 or above, 
- boto3 1.24 or above 
- botocore 1.27 or above 

In [3]:
!pip show sagemaker boto3 botocore |egrep 'Name|Version|---'

zsh:1: command not found: pip


### B. Run a heterogeneous cluster training job

#### Step 1: Set up training environment
Import the required libraries that enable you to use Heterogeneous clusters for training. In this step, you are also inheriting this notebook's IAM role and SageMaker session. 

In [4]:
import datetime

from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute
from sagemaker.core.shapes import InstanceGroup, StoppingCondition
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

sess = Session()
role = "arn:aws:iam::729646638167:role/SageMakerRole"  # [papermill-run] explicit role
region = "us-west-1"  # [papermill-run] explicit region


[07/09/26 13:32:41] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8878027;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8878028;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8878033;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8878034;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

#### Step 2: Define instance groups 
Here we define instance groups. Each instance group includes a different instance type.

In [5]:
data_group = InstanceGroup(
    instance_group_name="data_group", instance_type="ml.c5.xlarge", instance_count=1
)
dnn_group = InstanceGroup(
    instance_group_name="dnn_group", instance_type="ml.m4.xlarge", instance_count=1
)

#### Step 3: Review the "hello world" training code

In [6]:
!pygmentize source_dir/train.py

zsh:1: command not found: pygmentize


#### Step 4: Configure the ModelTrainer
In order to use SageMaker to run our training code, we create a `ModelTrainer` that defines how to use the container to train. In SageMaker Python SDK v3, framework estimators (such as the v2 `TensorFlow` estimator) are replaced by the generic `ModelTrainer` combined with a framework training image retrieved via `image_uris.retrieve`.

The heterogeneous cluster is configured through the `Compute` object by passing the list of instance groups. Note that when `instance_groups` is provided, you do not set `instance_type`/`instance_count` on `Compute`.

In [7]:
# Retrieve the TensorFlow training image (v2 used framework_version="2.9", py_version="py39")
training_image = image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="2.9",
    py_version="py39",
    instance_type="ml.c5.xlarge",
    image_scope="training",
)

source_code = SourceCode(
    source_dir="/Users/lucasjia/pysdk/pysdk-workspace/amazon-sagemaker-examples-lucas/      build_and_train_models/sm-heterogeneous_clusters_training/source_dir",  # [papermill-run] absolute path
    entry_script="train.py",
)

compute = Compute(
    instance_groups=[
        data_group,
        dnn_group,
    ],
    volume_size_in_gb=10,
)

model_trainer = ModelTrainer(
    training_image=training_image,
    source_code=source_code,
    compute=compute,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    role=role,
    base_job_name="hello-world-heterogenous",
)

[07/09/26 13:32:42] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=8878039;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=8878040;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=8878047;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8878048;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Role 'arn:aws:iam::729646638167:role/SageMakerRole' validated ]8;id=8878055;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=8878056;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             for training. Using it.                                                               

[07/09/26 13:32:43] INFO     OutputDataConfig not provided. Using default:                          ]8;id=8878062;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8878063;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-1-729646638167/hello-world-hete                
                             rogenous' kms_key_id=None compression_type='GZIP'                                     

                    INFO     Training image URI:                                               ]8;id=8878070;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8878071;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-1.amazonaws.com/tensorflow-training:                     
                             2.9-cpu-py39                                                                          

#### Step 5: Submit the training job
Here you are submitting the heterogeneous cluster training job. 

In [8]:
model_trainer.train()

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=8878078;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=8878079;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


[07/09/26 13:32:46] INFO     Creating training_job resource.                                     ]8;id=8878086;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8878087;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31123\31123]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=8878094;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=8878095;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

...

                    INFO     hello-world-heterogenous-20260709133243/algo-2-1783629216:          ]8;id=8879668;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8879669;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31469\31469]8;;\
                             env.distribution_hosts: TODO                                                          

                    INFO     hello-world-heterogenous-20260709133243/algo-2-1783629216:          ]8;id=8879674;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8879675;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31469\31469]8;;\
                             env.distribution_instance_groups: TODO                                                

                    INFO     hello-world-heterogenous-20260709133243/algo-2-1783629216:          ]8;id=8879680;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8879681;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31469\31469]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     hello-world-heterogenous-20260709133243/algo-2-1783629216:          ]8;id=8879686;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8879687;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31469\31469]8;;\
                             Training Container Execution Completed                                                

[07/09/26 13:35:58] INFO     Final Resource Status: Completed                                    ]8;id=8879693;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=8879694;file:///Users/lucasjia/pysdk/pysdk-workspace/_papermill_experiment/venv/lib/python3.12/site-packages/sagemaker/core/resources.py#31475\31475]8;;\

#### Step 6: Review the logs for environment information

Wait for the training job to finish, and review its logs in the AWS Console (click on **View logs** from the **Training Jobs** node in **Amazon SageMaker Console**)  You'll find two logs: Algo1, Algo2. Examine the printouts on each node on how to retrieve instance group environment information. An example is shown here:

```
Option-1: Read instance group information from the sagemaker_training.environment.Environment class
env.is_hetero: True
env.current_host: algo-1
env.current_instance_type: ml.c5.xlarge
env.current_instance_group: data_group
env.current_instance_group_hosts: ['algo-1']
env.instance_groups: ['data_group', 'dnn_group']

Option-2: Read instance group information from {file_path}.            You'll need to parse the json yourself. This doesn't require an additional library.
/opt/ml/input/config/resourceconfig.json dump = {
    "current_group_name": "data_group",
    "current_host": "algo-1",
    "current_instance_type": "ml.c5.xlarge",
    "hosts": [
        "algo-1",
        "algo-2"
    ],
    "instance_groups": [
        {
            "hosts": [
                "algo-1"
            ],
            "instance_group_name": "data_group",
            "instance_type": "ml.c5.xlarge"
        },
        {
            "hosts": [
                "algo-2"
            ],
            "instance_group_name": "dnn_group",
            "instance_type": "ml.m4.xlarge"
        }
    ],
    "network_interface_name": "eth0"
}
env.is_hetero: True
current_host=algo-1
current_instance_type=ml.c5.xlarge
env.current_instance_group: data_group
env.current_instance_group_hosts: TODO
env.instance_groups: TODO
env.instance_groups_dict: [{'instance_group_name': 'data_group', 'instance_type': 'ml.c5.xlarge', 'hosts': ['algo-1']}, {'instance_group_name': 'dnn_group', 'instance_type': 'ml.m4.xlarge', 'hosts': ['algo-2']}]
env.distribution_hosts: TODO
env.distribution_instance_groups: TODO
```

### C. Next steps

In this notebook, we demonstrated how to retrieve the environment information, and differentiate which instance group an instance belongs to. Based on this, you can build logic to offload data processing tasks in your training job to a dedicated instance group. To understand how that can be done with a real-world example, we suggest going through the following notebook examples:  

- [TensorFlow's tf.data.service based Amazon SageMaker Heterogeneous Clusters for training](../tf.data.service.sagemaker/hetero-tensorflow-restnet50.ipynb)
- [PyTorch and gRPC distributed dataloader based Amazon SageMaker Heterogeneous Clusters for training](../pt.grpc.sagemaker/hetero-pytorch-mnist.ipynb)

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)
